In [ ]:
# This example will demonstrate:

    # Defining a Custom Tool: How to create a Python class that encapsulates the logic for interacting with your MCP server.
    # Creating Agents: How to define AI agents with specific roles, goals, and backstories.
    # Defining Tasks: How to set up tasks that your agents will perform, utilizing the custom tools.
    # Orchestrating a Crew: How to bring agents, MCP and tasks together into a collaborative crew.

# --- 0. Import CrewAI & MCPServerAdapter ---

In [11]:
from crewai import Agent
from crewai_tools import MCPServerAdapter
from crewai import Task, Crew, LLM
from crewai import Agent, Task, Crew, Process, LLM

In [12]:
# 2. SSE Server:
# # MCP configuration for websearch and memory MCP servers
server_params = [
    {
        "url": "http://localhost:8070/sse",
        "transport": "sse"
    },
    {
    "url": "http://localhost:8000/sse",
    "transport": "sse"
    }
  ]

with MCPServerAdapter(server_params) as mcp_tools:
    print(f"Available tools: {[tool.name for tool in mcp_tools]}")

# This snippet sets up a connection to an MCP (Multi-Agent Control Protocol) Server using Server-Sent Events (SSE) 
# to dynamically load tools that agents can use in a CrewAI workflow.

# Agent: Represents an autonomous CrewAI agent that can perform tasks using tools.
# MCPServerAdapter: A connector that allows CrewAI to fetch tools from an external MCP server.

# url: Points to the local MCP server endpoint that streams tool metadata via SSE.
# transport: Specifies the protocol (sse) used for real-time communication.

# with MCPServerAdapter(...): Initializes the connection to the MCP server and automatically cleans up afterward.
# with MCPServerAdapter(server_params) as mcp_tools:: This is a with statement, a Python construct that ensures resources are properly managed.
# In this case, it handles the connection to the MCP server.
# mcp_tools: A list-like object containing tool instances fetched from the server.
# tool.name: Prints the name of each tool available to the agent.


Available tools: ['web_search', 'add_memory', 'search_memory_nodes', 'search_memory_facts', 'delete_entity_edge', 'delete_episode', 'get_entity_edge', 'get_episodes', 'clear_graph']


# --- 1. Import Opik ---

In [9]:
# Import opik and its CrewAI integration
import opik
from opik.integrations.crewai import track_crewai
track_crewai(project_name="arunmanglick-crewai-integration-demo")

# --- 1. Setup Local LLM ---

In [13]:
# The LLM class is used to configure a language model.
# We are specifying 'ollama/llama3.2' as the model and pointing to the default local Ollama server address.

local_llm = LLM(
    model="ollama/llama3.2",
    base_url="http://localhost:11434"
)

# What is Ollama
# Ollama is a platform that lets you run LLMs locally on your machine—no cloud dependency required. 
# It’s designed for developers who want fast, private, and customizable access to models like LLaMA, Mistral, Gemma, Phi-4, and more.


# --- 2. Define AI Agent ---

In [ ]:
# from crewai import LLM

# # Uncomment and configure your local LLM (Ollama) if not already done
# local_llm = LLM(
#     model="ollama/llama3.2",
#     base_url="http://localhost:11434"
# )

# This code to use local LLM is commented this time, as the code is using the LLM from OpenAI
# To use OpenAI, logged-in here https://platform.openai.com/ 
# Generated Key 
# Added the 'OPENAI_API_KEY' in env file
# ------------------------------------------------------------

with MCPServerAdapter(server_params) as mcp_tools:
    print(f"Available tools: {[tool.name for tool in mcp_tools]}")

    # Web Search Agent 
    web_search_agent = Agent(
        role="Web Search Agent",
        goal="Search the web for information",
        backstory="You are an agent that can search the web for information.",
        allow_delegation=False,
        tools=[mcp_tools["web_search"]],
        llm=local_llm  # Assign the local LLM to this agent
    )

    # Memory Agent
    memory_agent = Agent(
        role="Memory Agent",
        goal="Remember important information",
        backstory="You are an agent that can remember important information.",
        allow_delegation=False,
        tools=[mcp_tools["add_memory"]],
        llm=local_llm  # Assign the local LLM to this agent
    )

    # Response Generator Agent
    response_generator = Agent(
        role="Response Generator",
        goal="Generate coherent responses using memory context",
        backstory="I analyze memory nodes to generate contextually relevant responses.",
        allow_delegation=False,
        tools=[mcp_tools["search_memory_nodes"]],
        llm=local_llm  # Assign the local LLM to this agent
    )

    # Define all three tasks
    web_search_task = Task(
        description="Search the web for information about {query}.",
        agent=web_search_agent,
        expected_output="A concise answer to the user's query based on the search results.",
    )

    memory_task = Task(
        description="Store the query in memory: {query}.",
        agent=memory_agent,
        expected_output="Confirmation that the information has been stored.",
    )

    response_task = Task(
        description="Generate a response based for the {query} using the stored information in memory.",
        agent=response_generator,
        expected_output="A coherent response generated from memory context.",
    )

    crew = Crew(
        agents=[web_search_agent, memory_agent, response_generator],
        tasks=[web_search_task, memory_task, response_task],
        verbose=True
    )

    result = crew.kickoff(inputs={"query": "What is latest news on Vertex Inc USA?"})
    print(result)

Available tools: ['web_search', 'add_memory', 'search_memory_nodes', 'search_memory_facts', 'delete_entity_edge', 'delete_episode', 'get_entity_edge', 'get_episodes', 'clear_graph']


c:\Users\Arun.Manglick\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'items', 'anyOf', 'enum', 'properties'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6f9a5c57-80c4-45b0-8b83-e553ba194aa9                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Search Agent                                                                                        │
│                                                                                                                 │
│  Task: Search the web for information about What is latest news on Vertex Inc USA?.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web Search Agent                                                                                        │
│                                                                                                                 │
│  Thought: Thought: I need to specify the query to be searched                                                   │
│                                                                                                                 │
│  Using Tool: web_search                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"query\": \"latest news on Vertex Inc USA\"}"                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  answer="The latest news on Vertex, Inc. (NASDAQ: VERX), a leading global provider of indirect tax technology   │
│  solutions, includes:\n\n- Vertex announced its first quarter 2025 financial results on May 7, 2025.\n- The     │
│  company completed a strategic investment in Kintsugi, an AI-native startup focused on automating sales tax     │
│  compliance for small and mid-size businesses.\n- Vertex announced 65 new enhancements to its tax technology    │
│  portfolio, including integrations with SAP, Oracle, Coupa, and Shopify, plus new AI-powered tools like Vertex  │
│  Copilot and Vertex Express Returns for the U.S.\n- Key improvements include readiness for Brazil tax reform,   │
│  expanded tax content, enhanced VAT ID validation across 67+ countries, new e-invoicing capabilities, and       │
│  strengthened cloud presence with Vertex Tax Calculation on Azure.\n- Vertex executives participated in         │
│  industry conferences such as the 20th Annual Needham Technology, Media, & Consumer Conference.\n- The company  │
│  plans to release its second quarter 2025 financial results on August 6, 2025.\n\nThese updates reflect         │
│  Vertex's ongoing innovation and expansion in tax technology solutions to help businesses with global tax       │
│  compliance and automation." sources=[LinkupSource(name='News | Vertex, Inc.',                                  │
│  url='https://ir.vertexinc.com/news-releases', snippet='The Investor Relations website contains information     │
│  about Vertex, Inc.&#x27;s business for stockholders, potential investors, and financial analysts.\nVertex,     │
│  Inc. to Host 2025 Investor Day ... Please visit the Vertex corporate news room for non-financial press         │
│  releases.'), LinkupSource(name='News from Vertex Inc.',                                                        │
│  url='https://www.vertexinc.com/company/news/latest-news', snippet="Vertex, Inc. announced financial results    │
│  for its first quarter of 2025. ... Vertex&#x27;s research reveals strong support for e-invoicing mandates      │
│  among global tax and finance leaders. ... Vertex invests in AI-native startup Kintsugi to transform sales tax  │
│  compliance for small and mid-size businesses.\...                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()